# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

**Dataset title:** Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset Croissant schema metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name if hasattr(metadata, 'name') else 'N/A'}")
print(f"Description: {metadata.description if hasattr(metadata, 'description') else 'N/A'}")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}")
print(f"Date Published: {getattr(metadata, 'datePublished', 'N/A')}")


## 2. Data Overview
Review available record sets, fields, and their IDs. All references use their `@id` values as defined by the Croissant schema.

First, let's enumerate all available record sets in the dataset and inspect their fields.

In [ ]:
# List record sets and their fields (referenced by @id)
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '<no name>')}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    - Field @id: {field.get('@id', '?')}  (label: {field.get('name', '?')})")
            else:
                print(f"    - Field @id: {field}")
        print('')


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

**Note:** For this dataset, we'll load all available record sets and display their columns. If you want to select a particular record set for your analysis, use its `@id` as shown above.

In [ ]:
# Extract data from each record set using their @id
dataframes = {}
if not record_sets:
    print("No record sets to extract.")
else:
    for rs in record_sets:
        record_set_id = rs['@id']
        print(f"\nLoading records for RecordSet @id: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records.")
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())
        except Exception as e:
            print(f"Could not load records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

For demonstration, select a record set and a numeric field (using their `@id`s as referenced above) for example statistics and transformations.

In [ ]:
#--- Choose the record set and numeric field by their @id ---#
if not dataframes:
    print('No DataFrames loaded yet.')
else:
    # Select the first record set as an example
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"Using RecordSet @id: {selected_record_set_id}")
    df = dataframes[selected_record_set_id]

    # Try to detect numeric fields
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_cols:
        print('No numeric columns found for EDA in this record set.')
    else:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field (column) '@id': {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.90) if df[numeric_field_id].dtype != 'bool' else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:0.2f} (top 10%):")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field if any
        cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field_id = None
        for c in cat_cols:
            if df[c].nunique() > 1 and df[c].nunique() < len(df) // 2:
                group_field_id = c
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id} (mean of each group):")
            display(grouped_df)
        else:
            print('No suitable group field found for grouping.')


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. 

Below is a simple histogram of the chosen numeric field, and if a group field was found, a boxplot per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if 'df' in locals() and not df.empty and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 4))
    plt.hist(df[numeric_field_id].dropna(), bins=30, alpha=0.7)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()
else:
    print('No suitable data available for visualization.')

## 6. Conclusion
In this notebook, we have loaded the Mass Spectrometry dataset using `mlcroissant`, inspected available record sets, examined fields with their `@id`s, performed basic exploratory analysis, and visualized key fields. Use this notebook as a template for further, domain-specific analysis on the dataset as needed.